# NYC Hospital Quality — Data Cleaning

**Source:** CMS Hospital General Information (Care Compare, July 2026 release), downloaded from data.cms.gov.

**Goal:** reduce the national file (5,419 hospitals) to a clean, analysis-ready table of New York City hospitals, with documented decisions about missing data.


In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/Hospital_General_Information.csv')
print(df.shape)
df.head(3)

(5419, 38)


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,NaN,15,15,NaN,10,10,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,NaN,15,15,NaN,10,10,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,NaN,15,15,NaN,10,9,NaN


## Step 1 — Filter to New York City

NYC's five boroughs correspond to five counties: New York (Manhattan), Kings (Brooklyn), Bronx, Queens, and Richmond (Staten Island). We match on state and county, uppercasing county names to guard against inconsistent capitalization.

In [2]:
nyc_counties = ['NEW YORK', 'KINGS', 'BRONX', 'QUEENS', 'RICHMOND']

nyc = df[(df['State'] == 'NY') & (df['County/Parish'].str.upper().isin(nyc_counties))].copy()
print(nyc.shape)
nyc['County/Parish'].str.upper().value_counts()

(45, 38)


County/Parish
NEW YORK    16
KINGS       11
QUEENS       8
BRONX        7
RICHMOND     3
Name: count, dtype: int64

## Step 2 — Handle the missing star ratings

`Hospital overall rating` is stored as text and contains `"Not Available"` for hospitals CMS does not rate. Converting to numeric turns those into `NaN` (proper missing values) rather than zeros — a hospital without a rating is unrated, not a zero-star hospital.

In [3]:
nyc['rating_numeric'] = pd.to_numeric(nyc['Hospital overall rating'], errors='coerce')

print('Rated:', nyc['rating_numeric'].notna().sum(), '| Unrated:', nyc['rating_numeric'].isna().sum())
nyc.loc[nyc['rating_numeric'].isna(), ['Facility Name', 'Hospital Type']]

Rated: 35 | Unrated: 10


,Facility Name,Hospital Type
3235,NY EYE AND EAR INFIRMARY OF MOUNT SINAI,Acute Care Hospitals
3365,CREEDMOOR PSYCHIATRIC CENTER,Psychiatric
3366,NEW YORK STATE PSYCHIATRIC INSTITUTE,Psychiatric
3375,SOUTH BEACH PSYCHIATRIC CENTER,Psychiatric
3378,GRACIE SQUARE HOSP,Psychiatric
3381,BRONX PSYCHIATRIC CENTER,Psychiatric
3382,MANHATTAN PSYCHIATRIC CENTER,Psychiatric
3383,KIRBY FORENSIC PSYCHIATRIC CENTER,Psychiatric
3385,KINGSBORO PSYCHIATRIC HOSPITAL,Psychiatric
3389,NEW YORK CITY CHILDRENS PSYCH CENTER,Psychiatric


**Finding:** the 10 unrated hospitals are not missing at random — they are almost all psychiatric facilities (plus one eye-and-ear specialty hospital). CMS star ratings are built from general acute-care measures these facilities don't report. We keep them in the cleaned file (flagged via `is_rated`) but comparative rating analysis will scope to rated acute-care hospitals.

## Step 3 — Select and rename columns

Keep the columns the analysis needs, rename to consistent `snake_case` (friendlier for SQL and Python), and add a readable `borough` column.

In [4]:
borough_map = {'NEW YORK': 'Manhattan', 'KINGS': 'Brooklyn', 'BRONX': 'Bronx',
               'QUEENS': 'Queens', 'RICHMOND': 'Staten Island'}
nyc['borough'] = nyc['County/Parish'].str.upper().map(borough_map)
nyc['is_rated'] = nyc['rating_numeric'].notna()

keep = {
    'Facility ID': 'facility_id',
    'Facility Name': 'facility_name',
    'Address': 'address',
    'City/Town': 'city',
    'ZIP Code': 'zip_code',
    'borough': 'borough',
    'Hospital Type': 'hospital_type',
    'Hospital Ownership': 'ownership',
    'Emergency Services': 'emergency_services',
    'rating_numeric': 'overall_rating',
    'is_rated': 'is_rated',
    'Count of MORT Measures Better': 'mortality_better',
    'Count of MORT Measures No Different': 'mortality_no_different',
    'Count of MORT Measures Worse': 'mortality_worse',
    'Count of Safety Measures Better': 'safety_better',
    'Count of Safety Measures No Different': 'safety_no_different',
    'Count of Safety Measures Worse': 'safety_worse',
    'Count of READM Measures Better': 'readmission_better',
    'Count of READM Measures No Different': 'readmission_no_different',
    'Count of READM Measures Worse': 'readmission_worse',
}
clean = nyc[list(keep)].rename(columns=keep)
clean.head()

,facility_id,facility_name,address,city,zip_code,borough,hospital_type,ownership,emergency_services,overall_rating,is_rated,mortality_better,mortality_no_different,mortality_worse,safety_better,safety_no_different,safety_worse,readmission_better,readmission_no_different,readmission_worse
3201,330009,BRONXCARE HOSPITAL CENTER,1276 FULTON AVENUE,BRONX,10456,Bronx,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1.0,True,0,5,0,0,7,0,0,2,3
3204,330014,JAMAICA HOSPITAL MEDICAL CENTER,89TH AVENUE AND VAN WYCK EXPRESSWAY,JAMAICA,11418,Queens,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1.0,True,0,7,0,3,2,1,0,5,2
3205,330019,"NEW YORK COMMUNITY HOSPITAL OF BROOKLYN, INC.",2525 KINGS HIGHWAY,BROOKLYN,11229,Brooklyn,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1.0,True,0,7,0,0,4,0,0,5,3
3207,330024,MOUNT SINAI HOSPITAL,ONE GUSTAVE L LEVY PLACE,NEW YORK,10029,Manhattan,Acute Care Hospitals,Voluntary non-profit - Private,Yes,4.0,True,4,4,0,4,3,1,1,8,2
3209,330028,RICHMOND UNIVERSITY MEDICAL CENTER,355 BARD AVENUE,STATEN ISLAND,10310,Staten Island,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1.0,True,0,7,0,1,6,0,0,8,2


## Step 4 — Save the cleaned file

In [5]:
clean.to_csv('../data/cleaned/nyc_hospitals_cleaned.csv', index=False)
print('Saved', len(clean), 'hospitals,', clean.shape[1], 'columns')
clean.groupby('borough').agg(hospitals=('facility_id', 'count'),
                             avg_rating=('overall_rating', 'mean')).round(2)

Saved 45 hospitals, 20 columns


,hospitals,avg_rating
borough,,
Bronx,7,2.17
Brooklyn,11,1.80
Manhattan,16,3.73
Queens,8,2.17
Staten Island,3,2.00


## Cleaning decisions log

1. **Scope:** national file filtered to the 45 hospitals in NYC's five boroughs.
2. **Missing ratings:** `"Not Available"` converted to true missing values (`NaN`), never zero. The 10 unrated facilities are psychiatric/specialty hospitals — kept in the file, flagged with `is_rated`, excluded from rating comparisons.
3. **Columns:** trimmed 38 → 20, renamed to snake_case; added `borough` for readability.
4. **Raw data untouched:** all changes live in `data/cleaned/`; the raw CMS file is preserved as downloaded.